# ✈️ Aviation Intelligence: Investigating the Drivers of Flight Delays

## Research Objective
The goal of this investigation is to move beyond simple prediction and understand the **underlying statistical drivers** of aviation delays. We want to answer: *Is delay a matter of chance, or are there predictable patterns tied to time, distance, and carrier performance?*

---

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load the sampled data
DATA_PATH = 'data/processed/flights_sampled.parquet'
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} records for analysis.")

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/flights_sampled.parquet'

## 1. Initial Observation: The Distribution of Delays

The first thing I want to look at is the `ARRIVAL_DELAY`. I've noticed in the raw data that there are many negative values. This indicates flights arriving early, which is just as important to model as the delays themselves.

**Research Question:** How is the delay distributed? Is it a normal distribution, or is it heavily skewed by extreme outliers?

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df['ARRIVAL_DELAY'], bins=100, kde=True, color='skyblue')
plt.axvline(0, color='red', linestyle='--')
plt.title('Distribution of Arrival Delays (Minutes)')
plt.xlabel('Delay in Minutes')
plt.ylabel('Frequency')
plt.show()

print(f"Mean Delay: {df['ARRIVAL_DELAY'].mean():.2f}")
print(f"Median Delay: {df['ARRIVAL_DELAY'].median():.2f}")
print(f"Skewness: {df['ARRIVAL_DELAY'].skew():.2f}")

## 2. Hypothesis 1: The Temporal Factor (Time of Day)

**Hypothesis:** Delays are not distributed evenly throughout the day. I suspect that as the day progresses, delays accumulate (the 'cascading delay' effect), making evening flights riskier.

**Test:** I will group the data by the hour of scheduled departure and perform a Chi-Square test of independence to see if the probability of a delay is significantly different across different hours.

In [ ]:
# Preparing the data for the test
# We'll use 'SCHEDULED_DEPARTURE' which is in HHMM format
df['HOUR'] = (df['SCHEDULED_DEPARTURE'] // 100).astype(int)

# Create a contingency table: Hour vs is_delayed
contingency_table = pd.crosstab(df['HOUR'], df['is_delayed'])

# Chi-Square Test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

print(f"Chi-Square Statistic: {chi2:.2f}")
print(f"P-Value: {p:.4e}")

if p < 0.05:
    print("✅ Result: Statistically significant! The hour of departure DOES impact delay probability.")
else:
    print("❌ Result: Not significant. Time of day does not appear to be a primary driver of delays.")

# Visualization: Hourly Delay Probability
hourly_risk = df.groupby('HOUR')['is_delayed'].mean() * 100
plt.figure(figsize=(12, 6))
hourly_risk.plot(kind='line', marker='o', color='orange')
plt.title('Probability of Delay by Hour of Departure')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Delay Probability (%)')
plt.xticks(range(0, 24))
plt.show()

## 3. Hypothesis 2: The Airline Factor

**Hypothesis:** Not all airlines are created equal. Some carriers may have more efficient ground operations or better scheduling, making them statistically less prone to delays.

**Test:** I will perform a One-Way ANOVA to see if the mean delay time differs significantly across the different airlines.

In [ ]:
# ANOVA Test
model = ols('ARRIVAL_DELAY ~ C(AIRLINE)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("--- ANOVA Results ---")
print(anova_table)

p_val = anova_table['PR(>F)'][0]
if p_val < 0.05:
    print(f"\n✅ Result: Statistically significant (p={p_val:.4e}). Airline choice matters!")
else:
    print(f"\n❌ Result: Not significant (p={p_val:.4f}). Airline doesn't seem to impact delay magnitude.")

# Visualization: Delay spread by Airline
top_airlines = df['AIRLINE'].value_counts().nlargest(10).index
df_top = df[df['AIRLINE'].isin(top_airlines)]

plt.figure(figsize=(14, 7))
sns.boxplot(x='AIRLINE', y='ARRIVAL_DELAY', data=df_top)
plt.title('Arrival Delay Spread by Top 10 Airlines')
plt.ylabel('Delay in Minutes')
plt.show()

## 4. Hypothesis 3: The Distance Factor

**Hypothesis:** Longer flights might be more stable, or perhaps they are more prone to large delays due to complex routing. I want to see if there is a correlation between flight distance and the magnitude of arrival delay.

**Test:** I will use Pearson's correlation coefficient and a scatter plot with a regression line.

In [ ]:
correlation, p_val = stats.pearsonr(df['DISTANCE'], df['ARRIVAL_DELAY'])

print(f"Pearson Correlation Coefficient: {correlation:.4f}")
print(f"P-Value: {p_val:.4e}")

if abs(correlation) < 0.1:
    print("❌ Result: Weak or no correlation between distance and delay.")
else:
    print("✅ Result: Significant correlation detected.")

plt.figure(figsize=(12, 6))
sns.regplot(x='DISTANCE', y='ARRIVAL_DELAY', data=df.sample(min(len(df), 10000)), 
            scatter_kws={'alpha':0.1}, line_kws={'color':'red'})
plt.title('Correlation between Flight Distance and Arrival Delay')
plt.xlabel('Distance (miles)')
plt.ylabel('Arrival Delay (minutes)')
plt.show()

## 5. Summary of Scientific Findings

Based on this investigation, we can conclude:

1. **The Time Component:** [Insert finding from Hypothesis 1]
2. **The Airline Component:** [Insert finding from Hypothesis 2]
3. **The Distance Component:** [Insert finding from Hypothesis 3]

**Next Steps:**
- Move from exploratory analysis to formal feature engineering.
- Use these insights to create more predictive features for the LightGBM model.